# World Cup 2026 Predictor: EDA and Modelling Notebook

This notebook documents a reproducible workflow for exploring the historical football data and the current modelling logic used by the Flask app.

It covers:

- Loading and validating the project datasets.
- Exploratory analysis of match results, goals, tournaments, shootouts, and team activity.
- Name normalization and World Cup 2026 team focus sets.
- Elo-style modelling aligned with `match_model.py`.
- Simple holdout diagnostics for probability quality.
- 2026 group-stage ranking and knockout prediction helpers.

The notebook is intentionally project-aware: it imports local helpers from `match_model.py`, `app.py`, and `schedule_data.py` where useful.

## 1. Setup

Run this notebook from the repository root. If you opened it from another working directory, update `PROJECT_ROOT` below.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    plt = None

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False
    sns = None

try:
    from sklearn.metrics import mean_squared_error, log_loss
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'

print(f'Project root: {PROJECT_ROOT}')
print(f'Data directory exists: {DATA_DIR.exists()}')

## 2. Load Data

The main modelling dataset is `results.csv`. Shootouts help interpret tied knockout matches, and goalscorers provide optional player/scoring EDA.

In [ ]:
results = pd.read_csv(DATA_DIR / 'results.csv')
shootouts = pd.read_csv(DATA_DIR / 'shootouts.csv')
goalscorers = pd.read_csv(DATA_DIR / 'goalscorers.csv')
former_names = pd.read_csv(DATA_DIR / 'former_names.csv')

frames = {
    'results': results,
    'shootouts': shootouts,
    'goalscorers': goalscorers,
    'former_names': former_names,
}

summary = pd.DataFrame(
    [
        {
            'dataset': name,
            'rows': len(df),
            'columns': len(df.columns),
            'missing_cells': int(df.isna().sum().sum()),
        }
        for name, df in frames.items()
    ]
)
summary

In [ ]:
for name, df in frames.items():
    print(f'\n{name}')
    display(df.head(3))

## 3. Cleaning and Feature Basics

Create typed dates, numeric scores, result labels, goal totals, and calendar features. Rows with unavailable scores are excluded from modelling but can still be inspected separately.

In [ ]:
results_clean = results.copy()
results_clean['date'] = pd.to_datetime(results_clean['date'], errors='coerce')
results_clean['home_score'] = pd.to_numeric(results_clean['home_score'], errors='coerce')
results_clean['away_score'] = pd.to_numeric(results_clean['away_score'], errors='coerce')
results_clean['neutral'] = results_clean['neutral'].astype(str).str.upper().eq('TRUE')

scored = results_clean.dropna(subset=['date', 'home_score', 'away_score']).copy()
scored['home_score'] = scored['home_score'].astype(int)
scored['away_score'] = scored['away_score'].astype(int)
scored['total_goals'] = scored['home_score'] + scored['away_score']
scored['goal_diff_home'] = scored['home_score'] - scored['away_score']
scored['result'] = np.select(
    [scored['goal_diff_home'].gt(0), scored['goal_diff_home'].lt(0)],
    ['home_win', 'away_win'],
    default='draw',
)
scored['year'] = scored['date'].dt.year
scored['decade'] = (scored['year'] // 10) * 10

print(f'Rows with complete scores: {len(scored):,}')
print(f'Date range: {scored.date.min().date()} to {scored.date.max().date()}')
scored.head()

In [ ]:
quality_checks = pd.Series({
    'missing_dates': int(results_clean['date'].isna().sum()),
    'missing_home_score': int(results_clean['home_score'].isna().sum()),
    'missing_away_score': int(results_clean['away_score'].isna().sum()),
    'negative_home_scores': int((scored['home_score'] < 0).sum()),
    'negative_away_scores': int((scored['away_score'] < 0).sum()),
    'duplicate_rows': int(results.duplicated().sum()),
})
quality_checks

## 4. Exploratory Data Analysis

Start with high-level distributions: match volume over time, result balance, goal environment, tournament mix, and neutral-site effects.

In [ ]:
eda_overview = pd.Series({
    'matches': len(scored),
    'teams_observed': pd.concat([scored['home_team'], scored['away_team']]).nunique(),
    'tournaments': scored['tournament'].nunique(),
    'avg_goals_per_match': scored['total_goals'].mean(),
    'draw_rate': (scored['result'] == 'draw').mean(),
    'neutral_share': scored['neutral'].mean(),
    'home_win_rate_non_neutral': scored.loc[~scored['neutral'], 'result'].eq('home_win').mean(),
    'home_win_rate_neutral': scored.loc[scored['neutral'], 'result'].eq('home_win').mean(),
})
eda_overview.round(3)

In [ ]:
yearly = scored.groupby('year').agg(
    matches=('date', 'size'),
    avg_goals=('total_goals', 'mean'),
    draw_rate=('result', lambda s: (s == 'draw').mean()),
).reset_index()

yearly.tail(10)

In [ ]:
if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    yearly.plot(x='year', y='matches', ax=axes[0], legend=False, color='#0b6b5e')
    axes[0].set_title('International matches by year')
    axes[0].set_ylabel('Matches')
    yearly.plot(x='year', y='avg_goals', ax=axes[1], legend=False, color='#9e2444')
    axes[1].set_title('Average goals per match by year')
    axes[1].set_ylabel('Goals')
    plt.tight_layout()
else:
    yearly.tail(20)

In [ ]:
result_mix = scored['result'].value_counts(normalize=True).rename('share').to_frame()
tournament_mix = scored['tournament'].value_counts().head(15).rename_axis('tournament').reset_index(name='matches')

display(result_mix.round(3))
display(tournament_mix)

In [ ]:
if HAS_MATPLOTLIB:
    ax = tournament_mix.sort_values('matches').plot.barh(x='tournament', y='matches', figsize=(10, 6), color='#0b6b5e', legend=False)
    ax.set_title('Top tournaments by match count')
    ax.set_xlabel('Matches')
    ax.set_ylabel('')
    plt.tight_layout()

## 5. Team-Level EDA

This section identifies active teams, historical scoring patterns, and the subset of teams represented in the current 2026 app groups.

In [ ]:
team_long = pd.concat([
    scored[['date', 'year', 'home_team', 'home_score', 'away_score', 'tournament', 'neutral']].rename(
        columns={'home_team': 'team', 'home_score': 'goals_for', 'away_score': 'goals_against'}
    ).assign(side='home'),
    scored[['date', 'year', 'away_team', 'away_score', 'home_score', 'tournament', 'neutral']].rename(
        columns={'away_team': 'team', 'away_score': 'goals_for', 'home_score': 'goals_against'}
    ).assign(side='away'),
], ignore_index=True)
team_long['goal_diff'] = team_long['goals_for'] - team_long['goals_against']
team_long['win'] = team_long['goal_diff'].gt(0)
team_long['draw'] = team_long['goal_diff'].eq(0)
team_long['loss'] = team_long['goal_diff'].lt(0)

team_summary = team_long.groupby('team').agg(
    matches=('team', 'size'),
    first_year=('year', 'min'),
    last_year=('year', 'max'),
    goals_for=('goals_for', 'sum'),
    goals_against=('goals_against', 'sum'),
    avg_goal_diff=('goal_diff', 'mean'),
    win_rate=('win', 'mean'),
    draw_rate=('draw', 'mean'),
).sort_values('matches', ascending=False)

team_summary.head(15)

In [ ]:
recent_team_summary = team_long[team_long['year'] >= 2018].groupby('team').agg(
    matches=('team', 'size'),
    goals_for_per_match=('goals_for', 'mean'),
    goals_against_per_match=('goals_against', 'mean'),
    avg_goal_diff=('goal_diff', 'mean'),
    win_rate=('win', 'mean'),
).query('matches >= 20').sort_values('avg_goal_diff', ascending=False)

recent_team_summary.head(20)

## 6. Name Normalization and 2026 Team Focus

The app uses display names such as `Korea Republic`, while historical data may use names such as `South Korea`. Import the app groups and model aliases to line up the datasets.

In [ ]:
from app import WORLD_CUP_GROUPS
from match_model import NAME_ALIASES, TEAM_NAME_BY_CODE, build_team_ratings, poisson_expected_goals

wc_teams = pd.DataFrame(
    [
        {'group': group['id'], 'team': team['name'], 'code': team['code']}
        for group in WORLD_CUP_GROUPS
        for team in group['teams']
    ]
)
wc_teams.head()

In [ ]:
def canonical_team_name(name: str) -> str:
    return NAME_ALIASES.get(name, name)

team_long_norm = team_long.copy()
team_long_norm['team_canonical'] = team_long_norm['team'].map(canonical_team_name)

wc_name_lookup = wc_teams.set_index('team')['code'].to_dict()
wc_alias_coverage = wc_teams.assign(
    historical_name=wc_teams['code'].map(TEAM_NAME_BY_CODE),
    canonical_historical_name=lambda df: df['historical_name'].map(canonical_team_name),
)
wc_alias_coverage.head(12)

In [ ]:
wc_recent = team_long_norm[team_long_norm['team_canonical'].isin(wc_alias_coverage['canonical_historical_name']) & (team_long_norm['year'] >= 2018)]
wc_recent_summary = wc_recent.groupby('team_canonical').agg(
    matches=('team_canonical', 'size'),
    goals_for_per_match=('goals_for', 'mean'),
    goals_against_per_match=('goals_against', 'mean'),
    avg_goal_diff=('goal_diff', 'mean'),
    win_rate=('win', 'mean'),
).sort_values('avg_goal_diff', ascending=False)

wc_recent_summary.head(20)

## 7. Current Rating Model Outputs

The production helper `build_team_ratings()` returns the ratings consumed by the Flask app and browser-side win probability functions.

In [ ]:
team_codes = wc_teams['code'].tolist()
ratings = build_team_ratings(team_codes)
rating_df = wc_teams.assign(rating=wc_teams['code'].map(ratings)).sort_values('rating', ascending=False)
rating_df.head(20)

In [ ]:
if HAS_MATPLOTLIB:
    top_rating = rating_df.head(20).sort_values('rating')
    ax = top_rating.plot.barh(x='team', y='rating', figsize=(9, 7), legend=False, color='#0b6b5e')
    ax.set_title('Top 20 app ratings among 2026 teams')
    ax.set_xlabel('Rating')
    ax.set_ylabel('')
    plt.tight_layout()

## 8. Probability Helpers

These helpers mirror the browser-side logic in `static/app.js` and `static/guided.js`.

In [ ]:
def win_probability(team_rating: float, opponent_rating: float) -> float:
    return 1 / (1 + math.pow(10, (opponent_rating - team_rating) / 400))


def expected_scoreline(team_rating: float, opponent_rating: float) -> tuple[int, int]:
    team_xg, opponent_xg = poisson_expected_goals(team_rating, opponent_rating)
    return max(0, round(team_xg)), max(0, round(opponent_xg))


def compare_teams(team_a_code: str, team_b_code: str) -> dict:
    team_a = wc_teams.loc[wc_teams['code'] == team_a_code].iloc[0]
    team_b = wc_teams.loc[wc_teams['code'] == team_b_code].iloc[0]
    p_a = win_probability(ratings[team_a_code], ratings[team_b_code])
    g_a, g_b = expected_scoreline(ratings[team_a_code], ratings[team_b_code])
    return {
        'team_a': team_a['team'],
        'team_b': team_b['team'],
        'team_a_rating': ratings[team_a_code],
        'team_b_rating': ratings[team_b_code],
        'team_a_win_probability': p_a,
        'team_b_win_probability': 1 - p_a,
        'estimated_score': f'{g_a}-{g_b}',
        'predicted_winner': team_a['team'] if p_a >= 0.5 else team_b['team'],
    }

compare_teams('BRA', 'ARG')

## 9. Simple Model Diagnostics

The production model is an Elo-style expected-score model. This diagnostic walks through historical matches chronologically, records pre-match expectations, updates ratings, and evaluates recent holdout calibration.

This is not a replacement for the app model; it is a notebook-friendly way to evaluate whether rating differences contain signal.

In [ ]:
from match_model import (
    Result,
    actual_home_score,
    expected_score,
    fifa_points_to_elo,
    goal_margin_multiplier,
    tournament_weight,
    weighted_k_factor,
    load_results,
)

historical_results = load_results()
print(f'Loaded model-ready historical results: {len(historical_results):,}')

In [ ]:
def evaluate_elo_history(results_list, holdout_start_year=2018, base_rating=1500, home_advantage=55):
    ratings_local = {}
    latest_date = max(r.played_on for r in results_list)
    records = []

    for result in sorted(results_list, key=lambda r: r.played_on):
        home_rating = ratings_local.get(result.home_name, base_rating)
        away_rating = ratings_local.get(result.away_name, base_rating)
        p_home = expected_score(home_rating + (0 if result.neutral else home_advantage), away_rating)
        actual = actual_home_score(result)

        records.append({
            'date': result.played_on,
            'year': result.played_on.year,
            'home_team': result.home_name,
            'away_team': result.away_name,
            'home_rating_pre': home_rating,
            'away_rating_pre': away_rating,
            'p_home_expected_score': p_home,
            'actual_home_score': actual,
            'home_score': result.home_score,
            'away_score': result.away_score,
            'tournament': result.tournament,
            'neutral': result.neutral,
        })

        k_factor = weighted_k_factor(result, latest_date)
        margin = goal_margin_multiplier(abs(result.home_score - result.away_score))
        change = k_factor * margin * (actual - p_home)
        ratings_local[result.home_name] = round(home_rating + change)
        ratings_local[result.away_name] = round(away_rating - change)

    predictions = pd.DataFrame(records)
    holdout = predictions[predictions['year'] >= holdout_start_year].copy()
    holdout['squared_error'] = (holdout['p_home_expected_score'] - holdout['actual_home_score']) ** 2
    holdout['absolute_error'] = (holdout['p_home_expected_score'] - holdout['actual_home_score']).abs()
    return predictions, holdout

pred_history, holdout = evaluate_elo_history(historical_results)

pd.Series({
    'prediction_rows': len(pred_history),
    'holdout_rows_2018_plus': len(holdout),
    'holdout_brier_like_mse': holdout['squared_error'].mean(),
    'holdout_mae': holdout['absolute_error'].mean(),
}).round(4)

In [ ]:
calibration = holdout.assign(
    probability_bin=pd.cut(holdout['p_home_expected_score'], bins=np.linspace(0, 1, 11), include_lowest=True)
).groupby('probability_bin').agg(
    matches=('actual_home_score', 'size'),
    predicted=('p_home_expected_score', 'mean'),
    observed=('actual_home_score', 'mean'),
).reset_index()

calibration

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], '--', color='#9aa4b2', label='perfect calibration')
    ax.plot(calibration['predicted'], calibration['observed'], marker='o', color='#0b6b5e', label='model bins')
    ax.set_title('Holdout calibration, 2018 onward')
    ax.set_xlabel('Mean predicted home expected score')
    ax.set_ylabel('Mean observed home score')
    ax.legend()
    plt.tight_layout()

## 10. Group-Stage Prediction Table

The app ranks teams in each group by summing pairwise win probabilities against the other teams in that group.

In [ ]:
def group_strength_table(group):
    rows = []
    for team in group['teams']:
        opponents = [opponent for opponent in group['teams'] if opponent['code'] != team['code']]
        expected_points_proxy = sum(win_probability(ratings[team['code']], ratings[opponent['code']]) for opponent in opponents)
        rows.append({
            'group': group['id'],
            'team': team['name'],
            'code': team['code'],
            'rating': ratings[team['code']],
            'pairwise_probability_sum': expected_points_proxy,
        })
    return pd.DataFrame(rows).sort_values('pairwise_probability_sum', ascending=False)

group_predictions = pd.concat([group_strength_table(group) for group in WORLD_CUP_GROUPS], ignore_index=True)
group_predictions['predicted_group_rank'] = group_predictions.groupby('group')['pairwise_probability_sum'].rank(ascending=False, method='first').astype(int)
group_predictions.sort_values(['group', 'predicted_group_rank']).head(24)

In [ ]:
qualified_projection = group_predictions[group_predictions['predicted_group_rank'].le(3)].copy()
qualified_projection.sort_values(['predicted_group_rank', 'rating'], ascending=[True, False]).head(36)

## 11. Match Schedule and Export-Ready Prediction Rows

This section shows how to produce a schedule-level prediction table similar to the browser CSV export. Group-stage teams are known from `MATCH_SCHEDULE`; knockout teams depend on current UI state in the app, so this notebook produces group-stage rows directly and leaves knockout expansion to the app or a dedicated simulator.

In [ ]:
from schedule_data import MATCH_SCHEDULE

schedule = pd.DataFrame(MATCH_SCHEDULE)
team_name_to_code = dict(zip(wc_teams['team'], wc_teams['code']))
team_name_to_code.update({
    'Bosnia & Herzegovina': 'BIH',
    'Cape Verde': 'CPV',
    'Czech Republic': 'CZE',
    'DR Congo': 'COD',
    'Iran': 'IRN',
    'Ivory Coast': 'CIV',
    'South Korea': 'KOR',
    'Turkey': 'TUR',
    'United States': 'USA',
})

def schedule_prediction_row(row):
    team_1_code = team_name_to_code.get(row['team1'])
    team_2_code = team_name_to_code.get(row['team2'])
    if not team_1_code or not team_2_code:
        return pd.Series({
            'team_1_code': team_1_code,
            'team_2_code': team_2_code,
            'team_1_win_probability': np.nan,
            'team_2_win_probability': np.nan,
            'predicted_winner': None,
            'estimated_result': None,
        })
    p1 = win_probability(ratings[team_1_code], ratings[team_2_code])
    g1, g2 = expected_scoreline(ratings[team_1_code], ratings[team_2_code])
    winner = row['team1'] if p1 >= 0.5 else row['team2']
    return pd.Series({
        'team_1_code': team_1_code,
        'team_2_code': team_2_code,
        'team_1_win_probability': p1,
        'team_2_win_probability': 1 - p1,
        'predicted_winner': winner,
        'estimated_result': f'{g1}-{g2}',
    })

group_schedule_predictions = schedule[schedule['match'].le(72)].copy()
group_schedule_predictions = pd.concat(
    [group_schedule_predictions, group_schedule_predictions.apply(schedule_prediction_row, axis=1)],
    axis=1,
)

group_schedule_predictions.head(12)

In [ ]:
export_preview = group_schedule_predictions[[
    'match', 'stage', 'group', 'date', 'time', 'venue', 'team1', 'team2',
    'team_1_win_probability', 'team_2_win_probability', 'predicted_winner', 'estimated_result'
]].copy()
export_preview['team_1_win_probability'] = (export_preview['team_1_win_probability'] * 100).round(1)
export_preview['team_2_win_probability'] = (export_preview['team_2_win_probability'] * 100).round(1)
export_preview.head(20)

## 12. Candidate Modelling Extensions

The current app is intentionally deterministic and explainable. If you want to extend modelling, good next experiments are:

1. **Three-class outcome model**: home win / draw / away win with calibrated probabilities.
2. **Poisson goal model**: predict score distributions instead of rounded expected goals.
3. **Team form features**: rolling goals, rolling xG proxy, recent strength of schedule.
4. **Tournament-context features**: neutral site, confederation, tournament weight, knockout vs friendly.
5. **Scenario persistence**: save bracket state and exported result assumptions for reproducibility.

Keep the app-facing output contract simple: team names, ratings, probabilities, predicted winner, result estimate, and provenance of assumptions.

In [ ]:
# Optional: save notebook-derived group-stage prediction preview.
# Uncomment if you want a local CSV artifact.
# output_path = PROJECT_ROOT / 'data' / 'group_stage_prediction_preview.csv'
# export_preview.to_csv(output_path, index=False)
# output_path